#ARU ASSIGNMENT - IoT and Machine Learning at the Edge on Arm

##IMPORT LIBRARIES

In [1]:
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D, Dropout, Flatten, MaxPooling2D, BatchNormalization, Activation

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

import matplotlib.pyplot as plt
import matplotlib.image as mpimg

import numpy as np
import random, tempfile, zipfile

from PIL import Image

import pandas as pd
import json

import os

In [2]:
# Change this to the folder where your Cassava dataset is stored
dataset_path = "cassava-leaf-disease-classification"

train_images_path = os.path.join(dataset_path, "train_images")
train_csv_path = os.path.join(dataset_path, "train.csv")
label_map_path = os.path.join(dataset_path, "label_num_to_disease_map.json")

In [3]:
# Load the labels CSV file
df = pd.read_csv(train_csv_path)

# Display the first 5 rows
print(df.head())

# Show basic information
print("\nDataset shape:", df.shape)
print("\nColumns:", df.columns.tolist())

         image_id  label
0  1000015157.jpg      0
1  1000201771.jpg      3
2   100042118.jpg      1
3  1000723321.jpg      1
4  1000812911.jpg      3

Dataset shape: (21397, 2)

Columns: ['image_id', 'label']


In [4]:
# Load label-to-disease mapping
with open(label_map_path, "r") as f:
    label_map = json.load(f)

print("Label mapping:")
print(label_map)

Label mapping:
{'0': 'Cassava Bacterial Blight (CBB)', '1': 'Cassava Brown Streak Disease (CBSD)', '2': 'Cassava Green Mottle (CGM)', '3': 'Cassava Mosaic Disease (CMD)', '4': 'Healthy'}


In [5]:
# Check whether image files listed in CSV exist
missing_files = []

for image_name in df['image_id']:
    image_path = os.path.join(train_images_path, image_name)
    if not os.path.exists(image_path):
        missing_files.append(image_name)

print("Number of missing files:", len(missing_files))

if len(missing_files) > 0:
    print("Example missing files:", missing_files[:5])

Number of missing files: 0


In [6]:
IMG_SIZE = 64
CHANNELS = 3
NUM_CLASSES = 5

In [7]:
def load_and_preprocess_image(image_path, img_size=96):
    image = Image.open(image_path).convert("RGB")
    image = image.resize((img_size, img_size))
    image = np.array(image, dtype=np.float32) / 255.0
    return image

In [8]:
X = []
y = []

for _, row in df.iterrows():
    image_name = row['image_id']
    label = row['label']
    
    image_path = os.path.join(train_images_path, image_name)
    image = load_and_preprocess_image(image_path, IMG_SIZE)
    
    X.append(image)
    y.append(label)

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.int32)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (21397, 64, 64, 3)
y shape: (21397,)


In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape, y_train.shape)
print("Test set:", X_test.shape, y_test.shape)

Training set: (17117, 64, 64, 3) (17117,)
Test set: (4280, 64, 64, 3) (4280,)


In [10]:
unique, counts = np.unique(y_train, return_counts=True)
print("Training class distribution:", dict(zip(unique, counts)))

unique, counts = np.unique(y_test, return_counts=True)
print("Test class distribution:", dict(zip(unique, counts)))

Training class distribution: {np.int32(0): np.int64(870), np.int32(1): np.int64(1751), np.int32(2): np.int64(1909), np.int32(3): np.int64(10526), np.int32(4): np.int64(2061)}
Test class distribution: {np.int32(0): np.int64(217), np.int32(1): np.int64(438), np.int32(2): np.int64(477), np.int32(3): np.int64(2632), np.int32(4): np.int64(516)}


In [12]:
model = Sequential()

model.add(Conv2D(4, (3, 3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)))
model.add(MaxPooling2D(2, 2))

model.add(Conv2D(8, (3, 3), activation='relu'))
model.add(MaxPooling2D(2, 2))

model.add(Conv2D(12, (3, 3), activation='relu'))
model.add(MaxPooling2D(2, 2))

model.add(Flatten())

model.add(Dense(16, activation='relu'))
model.add(Dense(5, activation='softmax'))

C:\Users\nickh\anaconda3\envs\ml_lab\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [13]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [13]:
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_test, y_test)
)

Epoch 1/10
535/535 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.5896 - loss: 1.1997 - val_accuracy: 0.6273 - val_loss: 1.1051
Epoch 2/10
535/535 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.6323 - loss: 1.0705 - val_accuracy: 0.6241 - val_loss: 1.0645
Epoch 3/10
535/535 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.6337 - loss: 1.0432 - val_accuracy: 0.6322 - val_loss: 1.0308
Epoch 4/10
535/535 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.6344 - loss: 1.0197 - val_accuracy: 0.6357 - val_loss: 1.0211
Epoch 5/10
535/535 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.6398 - loss: 1.0037 - val_accuracy: 0.6297 - val_loss: 1.0302
Epoch 6/10
535/535 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.6403 - loss: 1.0058 - val_accuracy: 0.6430 - val_loss: 0.9978
Epoch 7/10
535/535 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.6520 - loss: 0.9647 - val_accuracy: 0.6435 - val_loss: 0.9961
Epoch 8/10
535/535 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.6419 - loss: 0.9760 - val_accuracy: 0.

In [14]:
loss, accuracy = model.evaluate(X_test, y_test)
print("Test Accuracy:", accuracy)

134/134 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6540 - loss: 0.9757
Test Accuracy: 0.6521028280258179


In [15]:
y_pred = np.argmax(model.predict(X_test), axis=1)

134/134 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


In [16]:
print("Precision:", precision_score(y_test, y_pred, average='weighted'))
print("Recall:", recall_score(y_test, y_pred, average='weighted'))
print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Precision: 0.5582350013781766
Recall: 0.6521028037383177
F1 Score: 0.5734864623153721

Classification Report:

              precision    recall  f1-score   support

           0       0.32      0.16      0.21       217
           1       0.37      0.22      0.28       438
           2       0.24      0.01      0.02       477
           3       0.71      0.97      0.82      2632
           4       0.36      0.17      0.24       516

    accuracy                           0.65      4280
   macro avg       0.40      0.31      0.31      4280
weighted avg       0.56      0.65      0.57      4280



In [17]:
model.save("cassava_edge_baseline_model.h5")

In [18]:
def get_model_size(file):
    size_bytes = os.path.getsize(file)
    size_mb = size_bytes / (1024 * 1024)
    return size_mb

print("Model size (MB):", get_model_size("cassava_edge_baseline_model.h5"))

Model size (MB): 0.13985443115234375


In [19]:
def representative_data_gen():
    for i in range(100):
        yield [X_train[i:i+1].astype(np.float32)]

# Load the smaller edge baseline model
model = tf.keras.models.load_model("cassava_edge_baseline_model.h5")

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_model_quant = converter.convert()

with open("cassava_edge_model_int8.tflite", "wb") as f:
    f.write(tflite_model_quant)

print("INT8 model saved successfully.")

INFO:tensorflow:Assets written to: C:\Users\nickh\AppData\Local\Temp\tmpm_w370ln\assets


INFO:tensorflow:Assets written to: C:\Users\nickh\AppData\Local\Temp\tmpm_w370ln\assets


Saved artifact at 'C:\Users\nickh\AppData\Local\Temp\tmpm_w370ln'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 64, 64, 3), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 5), dtype=tf.float32, name=None)
Captures:
  1643480993856: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1643480994032: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1643481258640: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1643481258288: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1643481354080: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1643481370864: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1643481524528: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1643481524352: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1643481585616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1643481585440: TensorSpec(shape=(), dtype=tf.resource, name=None)


C:\Users\nickh\anaconda3\envs\ml_lab\lib\site-packages\tensorflow\lite\python\convert.py:854: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


INT8 model saved successfully.


In [20]:
path_tflite_model = "models/optimized_models/cassava_edge_model_dynamic.tflite"

tflite_interpreter = tf.lite.Interpreter(model_path=path_tflite_model)

input_details = tflite_interpreter.get_input_details()
output_details = tflite_interpreter.get_output_details()

print("== Input details ==")
print("Name:", input_details[0]['name'])
print("Shape:", input_details[0]['shape'])
print("Type:", input_details[0]['dtype'])
print("Quantisation scale:", input_details[0]['quantization'][0])
print("Zero point:", input_details[0]['quantization'][1])

print("\n== Output details ==")
print("Name:", output_details[0]['name'])
print("Shape:", output_details[0]['shape'])
print("Type:", output_details[0]['dtype'])
print("Quantisation scale:", output_details[0]['quantization'][0])
print("Zero point:", output_details[0]['quantization'][1])

== Input details ==
Name: serving_default_input_layer:0
Shape: [ 1 64 64  3]
Type: <class 'numpy.int8'>
Quantisation scale: 0.003921568859368563
Zero point: -128

== Output details ==
Name: StatefulPartitionedCall_1:0
Shape: [1 5]
Type: <class 'numpy.int8'>
Quantisation scale: 0.00390625
Zero point: -128


C:\Users\nickh\anaconda3\envs\ml_lab\lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [21]:
def get_gzipped_model_size(file):
    # Returns size of gzipped model, in bytes.
    _, zipped_file = tempfile.mkstemp('.zip')
    with zipfile.ZipFile(zipped_file, 'w', compression=zipfile.ZIP_DEFLATED) as f:
        f.write(file)
    return os.path.getsize(zipped_file)

fp_model_path = "cassava_edge_baseline_model.h5"
quant_model_path = "cassava_edge_model_int8.tflite"

fp_size = get_gzipped_model_size(fp_model_path)
quant_size = get_gzipped_model_size(quant_model_path)

print("Size of Full Precision Model: {} Bytes".format(fp_size))
print("Size of Quantized Model: {} Bytes".format(quant_size))
print("Size reduction factor: {:.2f}x".format(fp_size / quant_size))

Size of Full Precision Model: 94493 Bytes
Size of Quantized Model: 10602 Bytes
Size reduction factor: 8.91x


In [22]:
# Allocate tensors (keep this active)
tflite_interpreter.allocate_tensors()

# Get quantization parameters
input_scale, input_zero_point = input_details[0]["quantization"]

# Store predictions for ALL test samples
predictions = np.zeros(len(X_test), dtype=int)

# Run inference on TFLite model
for i in range(len(X_test)):
    val = X_test[i]

    # Apply quantization if needed
    if input_scale != 0:
        val = val / input_scale + input_zero_point

    # Match expected input format
    val = np.expand_dims(val, axis=0).astype(input_details[0]["dtype"])

    tflite_interpreter.set_tensor(input_details[0]['index'], val)
    tflite_interpreter.invoke()

    output = tflite_interpreter.get_tensor(output_details[0]['index'])

    predictions[i] = np.argmax(output)

# Compute accuracy (correct for integer labels)
correct = np.sum(predictions == y_test)
quant_accuracy = correct / len(y_test)


# Load full precision model
fp_model = tf.keras.models.load_model("cassava_edge_baseline_model.h5")

# Evaluate full precision model
loss, fp_accuracy = fp_model.evaluate(X_test, y_test, verbose=0)

# Print results
print("Accuracy of quantized INT8 model: {:.2f}%".format(quant_accuracy * 100))
print("Accuracy of full precision model: {:.2f}%".format(fp_accuracy * 100))
print("Accuracy change: {:.2f}%".format((quant_accuracy - fp_accuracy) * 100))

Accuracy of quantized INT8 model: 65.30%
Accuracy of full precision model: 65.21%
Accuracy change: 0.09%


In [15]:
import numpy as np

np.save("X_train.npy", X_train)
np.save("X_test.npy", X_test)
np.save("y_test.npy", y_test)